# 04b — Is mouse 30-M-2 sick? (Lung only)

A detour from `04_composition`: Lung mouse 30-M-2 supplies most of the myeloid cells. This notebook asks whether it is a healthy old mouse or one with a myeloid condition, which decides the exclusion in `ag.TISSUE_SETTINGS["Lung"]`.

Reads `data/Lung/processed.h5ad`.

### 1. Setup

In [ ]:
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from src import aging as ag

TISSUE = "Lung"
P = ag.paths(TISSUE)
adata = sc.read_h5ad(P.processed)
meta = adata.obs[["mouse.id", "age", "sex"]].drop_duplicates().set_index("mouse.id")

### 2. How much does 30-M-2 contribute to each cell type?

If one mouse supplies most of a cell type, any "age effect" there may be that mouse.

In [ ]:
share = pd.crosstab(adata.obs["cell_ontology_class"], adata.obs["mouse.id"], normalize="index")
print(share["30-M-2"].sort_values(ascending=False).head(10).round(2))

### 3. Is 30-M-2 sick? Monocyte-lineage share per mouse

A 30-month mouse (about 80+ human years) with a huge monocyte expansion could have an infection, a tumor, or another inflammatory condition. There are no health records, so the cells' genes have to show which explanation fits.

In [ ]:
# Monocyte lineage only: narrower than ag.MYELOID, which also has alveolar macrophages
myeloid = ["classical monocyte", "intermediate monocyte", "non-classical monocyte", "lung macrophage"]
tab = pd.crosstab(adata.obs["mouse.id"], adata.obs["cell_ontology_class"])
summary = pd.DataFrame({
    "total_cells": tab.sum(axis=1),
    "myeloid_cells": tab[myeloid].sum(axis=1),
})
summary["myeloid_fraction"] = (summary.myeloid_cells / summary.total_cells).round(2)
summary = summary.join(meta).sort_values("myeloid_fraction", ascending=False)
summary.to_csv(P.figure("myeloid_by_mouse.csv"))
summary

### 4. What state are the classical monocytes in, per mouse?

| Group | Genes | High in the outlier mouse would suggest |
| --- | --- | --- |
| Monocyte identity | *Ly6c2*, *Ccr2* | Ordinary classical monocytes |
| Interferon response | *Ifit1*, *Ifit3*, *Isg15*, *Irf7* | Antiviral-type response (e.g., infection) |
| Inflammation | *S100a8*, *S100a9*, *Il1b* | Acute inflammation or emergency myelopoiesis |
| Proliferation | *Mki67*, *Top2a* | Cells actively dividing (expansion or a myeloid tumor) |
| Blood contamination | *Hba-a1*, *Beta-s* | Lung not cleared of blood, circulating cells captured |

In [ ]:
mono = adata[adata.obs.cell_ontology_class == "classical monocyte"]
state_genes = {
    "identity": ["Ly6c2", "Ccr2"],
    "interferon": ["Ifit1", "Ifit3", "Isg15", "Irf7"],
    "inflammation": ["S100a8", "S100a9", "Il1b"],
    "proliferation": ["Mki67", "Top2a"],
    "blood": ["Hba-a1", "Beta-s"],
}
state_genes = {k: [g for g in v if g in adata.var_names] for k, v in state_genes.items()}

In [ ]:
sc.pl.dotplot(mono, state_genes, groupby="mouse.id", standard_scale="var", show=False)
plt.savefig(P.figure("monocyte_state.png"), dpi=200, bbox_inches="tight")

### 5. What makes 30-M-2's monocytes different from other old mice's?

Exploratory only: per-cell tests overstate significance, but the top gene list is still informative.

In [ ]:
mono_old = mono[mono.obs.age == "30m"].copy()
mono_old.obs["is_30M2"] = np.where(mono_old.obs["mouse.id"] == "30-M-2", "30-M-2", "other 30m")
sc.tl.rank_genes_groups(mono_old, groupby="is_30M2", groups=["30-M-2"], reference="other 30m", method="wilcoxon")
top30 = sc.get.rank_genes_groups_df(mono_old, group="30-M-2").head(30)
top30.to_csv(P.figure("30M2_top_genes.csv"), index=False)
top30

### 6. Is 30-M-2 different in every cell type, or only myeloid cells?

30-M-2 has few non-myeloid cells, so only read these for cell types where it has at least ~20 cells. If its B and T cells also show the signal, the condition is body-wide. If they look normal, it is specific to myeloid cells.

In [ ]:
genes_flat = [g for v in state_genes.values() for g in v]
for ct in ["B cell", "CD8-positive, alpha-beta T cell", "NK cell"]:
    sub = adata[(adata.obs.cell_ontology_class == ct) & (adata.obs.age.isin(["18m", "21m", "30m"]))]
    sc.pl.dotplot(sub, genes_flat, groupby="mouse.id", standard_scale="var", title=ct, show=False)
    name = ct.split(",")[0].replace(" ", "_")
    plt.savefig(P.figure(f"state_{name}.png"), dpi=200, bbox_inches="tight")

### Findings

- **30-M-2 is 97% myeloid** (7,789 of 8,014 cells). It holds 92% of all lung macrophages, 77% of intermediate monocytes, and 68% of classical monocytes. The next highest mouse is 30-M-3 at 51%.
- **Its classical monocytes are in an unusual state:** much lower *Ccr2* and lower *Ly6c2*, very high *S100a8/9*, some *Mki67* and *Top2a*, and no interferon signal.
- **Interpretation:** an expanding, inflammatory, less mature myeloid population. This fits an inflammatory condition or a myeloid disorder; the data can't say which. **Decision: exclude 30-M-2 from the myeloid cell types and from composition trends** (`ag.TISSUE_SETTINGS["Lung"]`, `ag.main_kw`).
- **Ambient RNA:** *S100a8/9* appear in B cells of 18-M-52, 18-M-53, and 30-M-2. B cells don't make these, so the reads likely come from free-floating RNA. Handled in Phase 5.